<a href="https://colab.research.google.com/github/Liularina/TexttoImage/blob/main/NSFW_IMAGE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio torch diffusers transformers accelerate

In [ ]:
import gradio as gr
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

# 1. 加载模型（使用 FP16 节省显存）
pipe = StableDiffusionXLPipeline.from_pretrained(
    "Heartsync/NSFW-Uncensored",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
)
pipe = pipe.to("cuda")

# 2. 优化：更换采样器为 Euler Ancestral（可选，提升画质）
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

# 3. 优化：启用注意力切片，降低显存占用（速度略降但更稳定）
pipe.enable_attention_slicing()

# 4. 定义生成函数
def generate_image(prompt, negative_prompt="", steps=30, guidance_scale=7.5):
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
    ).images[0]
    return image

# 5. 创建 Gradio 界面
iface = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Textbox(label="Prompt", lines=2, placeholder="描述你想要的画面..."),
        gr.Textbox(label="Negative Prompt", lines=1, placeholder="不想出现的元素..."),
        gr.Slider(10, 50, value=30, label="Steps (步数)"),
        gr.Slider(1, 15, value=7.5, label="Guidance Scale (提示词相关性)")
    ],
    outputs=gr.Image(label="生成的图像"),
    title="Heartsync NSFW 图像生成 API (优化版)",
    description="基于 Heartsync/NSFW-Uncensored，使用 Euler Ancestral 采样器 + Attention Slicing"
)

# 6. 启动服务（share=True 生成公网链接）
iface.launch(share=True)

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/712 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://63d73b049b6f7fda38.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
